---
title: 'Aufgabenblatt 3: Numerische Lösung'
engine: julia
---

In [ ]:
using DotMaps
using IntervalSets
using MMJMesh.Plots
using MMJMesh.Meshes
using MMJMesh.Mathematics
using CairoMakie: Figure, Axis, scatterlines, scatterlines!

## Problemparameter

In [ ]:
# Mehrfachvariable
p = DotMap()

# Basiswerte
p.E = 35e9
p.d = 0.8
p.l = 20
p.F = 2e6
p.ρ = 2500
p.g = 9.81
p.S = 120e6
p.c = 1.75e7

# Abgeleitete Werte
p.Ω = 0 .. p.l
p.C = p.c * π * p.d
p.A = π * p.d^2 / 4
p.EA = p.E * p.A
p.n = p.ρ * p.g * p.A;

## Exakte Lösung

In [ ]:
a1 = 3 + 7π
a2 = 7π - 3
x = parameter(p.Ω)

u = exp(-x / 20) / (3500000 * π * (a1 * ℯ^2 - a2)) * (
    25000 * (a1 * ℯ^2 + a2 * exp(x / 10))
    -
    2943 * π * ℯ * (exp(x / 10) + 1)
    +
    981 * π * exp(x / 20) * (a1 * ℯ^2 - a2)
)

N = p.EA * u'

null = 0 * x

fig = Figure()
fplot!(Axis(fig[1, 1], title="Verschiebung u"), u, null, npoints=100)
fplot!(Axis(fig[2, 1], title="Normalkraft N"), N, null, npoints=100)
fig

## Numerische Lösung

In [ ]:
include("src/fem.jl")
include("src/pile.jl")

### Ein Netz

In [ ]:
m = Mesh(0 .. 20, 40)
setdata!(group(m, :elements), :ke_func, pile_ke(p.EA, p.C))
setdata!(group(m, :elements), :re_func, pile_re(p.n))
setdata!(node(m, nnodes(m)), :ke_func, spring(p.S))
setdata!(node(m, 1), :re_func, force(p.F))
K, r = assemble_kr(m)
u_hat = K \ r

mplot(m, u_hat, nodesvisible=false) |> mconf()

### Konvergenz

In [ ]:
function pile_fem(p, k)
    m = Mesh(p.Ω, k)
    setdata!(group(m, :elements), :ke_func, pile_ke(p.EA, p.C))
    setdata!(group(m, :elements), :re_func, pile_re(p.n))
    setdata!(node(m, nnodes(m)), :ke_func, spring(p.S))
    setdata!(node(m, 1), :re_func, force(p.F))
    K, r = assemble_kr(m)
    u_hat = K \ r
    uh = sum(u_hat .* hatfunctions(range(p.Ω, k + 1)))
    return uh
end

uh = pile_fem(p, 3)
fplot(uh, u, npoints=100)

In [ ]:
Nh = p.EA * uh'
fplot(Nh, N, npoints=100)

In [ ]:
fplot(0 * x, u - uh, npoints=100)

## Konvergenz

Fehlerfunktion

In [ ]:
fig = Figure()
Axis(fig[1, 1])
for k = 2 .^ (1:5)
    uh = pile_fem(p, k)
    fplot!(u - uh, npoints=100)
end
fig

Normalkraft

In [ ]:
fig = Figure()
Axis(fig[1, 1])
fplot!(N)
for k = 2 .^ (1:5)
    uh = pile_fem(p, k)
    Nh = p.EA * uh'
    fplot!(Nh, connect_jumps=true)
end
fig

Fehler in max-Norm

In [ ]:
nn = 12
hks = zeros(nn)
eks = zeros(nn)

maxnorm(u) = maximum(abs.(u.(range(domain(u), 1000))))

for i in 1:nn
    k = 2^i
    uh = pile_fem(p, k)
    hks[i] = p.l / k
    eks[i] = 100 * maxnorm(u - uh) / maxnorm(u)
end

In [ ]:
scatterlines(hks, eks)

In [ ]:
f = Figure()
Axis(f[1, 1], xscale=log10, yscale=log10, title="Fehler in Prozent")
scatterlines!(hks, eks)
f